In [ ]:
%pip install -q Flask Flask-SQLAlchemy Flask-Session Flask-WTF argon2-cffi cachelib


In [ ]:
from flask import Flask, request, jsonify, session
from flask_sqlalchemy import SQLAlchemy
from flask_session import Session
from flask_wtf.csrf import CSRFProtect, generate_csrf
from argon2 import PasswordHasher
from argon2.exceptions import HashingError, VerifyMismatchError, VerificationError
from datetime import datetime, timedelta, timezone
from cachelib import SimpleCache
import secrets


app = Flask(__name__)
app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///auth.db"
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False
app.config["SECRET_KEY"] = secrets.token_hex(32)  # Prototype only; set persistent secret via environment for deployment
app.config["SESSION_COOKIE_HTTPONLY"] = True
app.config["SESSION_COOKIE_SECURE"] = True
app.config["SESSION_COOKIE_SAMESITE"] = "Lax"
app.config["PERMANENT_SESSION_LIFETIME"] = timedelta(minutes=30)
app.config["SESSION_TYPE"] = "cachelib"
app.config["SESSION_CACHELIB"] = SimpleCache()  # Prototype only; replace with shared persistent session store for deployment
app.config["SESSION_PERMANENT"] = True
app.config["SESSION_REFRESH_EACH_REQUEST"] = True
app.config["SESSION_SERIALIZATION_FORMAT"] = "json"
Session(app)
db = SQLAlchemy(app)
csrf = CSRFProtect(app)
password_hasher = PasswordHasher(time_cost=2, memory_cost=19456, parallelism=1, type=__import__('argon2').Type.ID)

@app.get("/api/auth/csrf-token")
def get_csrf_token():
    return jsonify({"csrf_token": generate_csrf()})


class User(db.Model):
    __tablename__ = "users"

    id = db.Column(db.Integer, primary_key=True)
    display_name = db.Column(db.String(50), nullable=False)
    email = db.Column(db.String(255), unique=True, nullable=False)
    password_hash = db.Column(db.String(255), nullable=False)
    birth_year = db.Column(db.Integer, nullable=False)
    weight = db.Column(db.Float, nullable=False)

    failed_login_attempts = db.Column(
        db.Integer, default=0, nullable=False
    )

    locked_until = db.Column(
        db.DateTime(timezone=True), nullable=True
    )

    privacy_notice_version = db.Column(
        db.String(20), nullable=False
    )

    privacy_accepted_at = db.Column(
        db.DateTime(timezone=True), nullable=False
    )


class SecurityLog(db.Model):
    __tablename__ = "security_logs"

    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer, nullable=True)
    event_type = db.Column(db.String(50), nullable=False)

    timestamp = db.Column(
        db.DateTime(timezone=True),
        default=lambda: datetime.now(timezone.utc),
        nullable=False
    )


with app.app_context():
    db.create_all()

print("User and SecurityLog tables created successfully!")

def log_security_event(event_type, user_id=None):
    log = SecurityLog(
        user_id=user_id,
        event_type=event_type
    )

    db.session.add(log)
    db.session.commit()

    print("Security event logged:", event_type)

@app.route("/api/auth/register", methods=["POST"])
def register():
    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({"error": "Invalid JSON"}), 400

    required = [
        "display_name", "email", "password",
        "birth_year", "weight", "privacy_consent"
    ]

    if any(field not in data for field in required):
        return jsonify({"error": "Missing required fields"}), 400

    name = data["display_name"]
    email = data["email"]
    password = data["password"]
    birth_year = data["birth_year"]
    weight = data["weight"]
    consent = data["privacy_consent"]

    # Validate display name
    if not isinstance(name, str) or not 1 <= len(name.strip()) <= 50:
        return jsonify({"error": "Invalid display name"}), 400

    # Normalize email
    if isinstance(email, str):
        email = email.strip().lower()

    # Validate email
    if not isinstance(email, str) or not (
        3 <= len(email) <= 255
        and email.count("@") == 1
        and "." in email.split("@")[-1]
        and not any(c.isspace() for c in email)
    ):
        return jsonify({"error": "Invalid email"}), 400

    # Validate password
    if not isinstance(password, str) or not 8 <= len(password) <= 128:
        return jsonify({
            "error": "Password must be 8-128 characters"
        }), 400

    # Validate birth year
    current_year = datetime.now(timezone.utc).year

    if (
        type(birth_year) is not int
        or not current_year - 100 <= birth_year <= current_year - 18
    ):
        return jsonify({"error": "Invalid birth year"}), 400

    # Validate weight
    if (
        type(weight) not in (int, float)
        or not 20 <= weight <= 300
    ):
        return jsonify({"error": "Invalid weight"}), 400

    # Validate privacy consent
    if consent is not True:
        return jsonify({
            "error": "Privacy consent is required"
        }), 400

    # Check duplicate email
    if User.query.filter_by(email=email).first():
        return jsonify({
            "error": "Email already registered"
        }), 409

    # Hash password using Argon2id
    try:
        hashed_password = password_hasher.hash(password)

    except HashingError:
        return jsonify({
            "error": "Registration failed"
        }), 500

    # Create user
    user = User(
        display_name=name.strip(),
        email=email,
        password_hash=hashed_password,
        birth_year=birth_year,
        weight=weight,
        failed_login_attempts=0,
        privacy_notice_version="1.0",
        privacy_accepted_at=datetime.now(timezone.utc)
    )

    db.session.add(user)
    db.session.commit()

    return jsonify({
        "message": "Account registered successfully",
        "user_id": user.id
    }), 201

@app.route("/api/auth/login", methods=["POST"])
def login():
    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({"error": "Invalid JSON"}), 400

    email = data.get("email")
    password = data.get("password")

    if not isinstance(email, str) or not isinstance(password, str):
        return jsonify({"error": "Email and password are required"}), 400

    email = email.strip().lower()
    user = User.query.filter_by(email=email).first()

    if user is None:
        log_security_event("LOGIN_FAILED")
        return jsonify({"error": "Invalid email or password"}), 401

    now = datetime.now(timezone.utc)
    locked_until = user.locked_until

    if locked_until is not None:
        if locked_until.tzinfo is None:
            locked_until = locked_until.replace(tzinfo=timezone.utc)

        if now < locked_until:
            log_security_event("LOGIN_BLOCKED", user.id)
            return jsonify({
                "error": "Account temporarily locked. Try again later."
            }), 423

        user.locked_until = None
        user.failed_login_attempts = 0
        db.session.commit()

    try:
        password_hasher.verify(user.password_hash, password)

    except (VerifyMismatchError, VerificationError):
        user.failed_login_attempts += 1

        if user.failed_login_attempts >= 5:
            user.locked_until = now + timedelta(minutes=15)

        db.session.commit()

        log_security_event("LOGIN_FAILED", user.id)

        return jsonify({
            "error": "Invalid email or password"
        }), 401

    user.failed_login_attempts = 0
    user.locked_until = None
    db.session.commit()

    session.clear()
    session["user_id"] = user.id
    session.permanent = True

    app.session_interface.regenerate(session)

    log_security_event("LOGIN_SUCCESS", user.id)

    return jsonify({
        "message": "Login successful"
    }), 200

@app.route("/api/auth/logout", methods=["POST"])
def logout():
    user_id = session.get("user_id")

    if user_id is None:
        return jsonify({
            "error": "Authentication required"
        }), 401

    log_security_event("LOGOUT", user_id)

    session.clear()

    return jsonify({
        "message": "Logout successful"
    }), 200

@app.route("/api/auth/change-credentials", methods=["POST"])
def change_credentials():
    user_id = session.get("user_id")

    if user_id is None:
        log_security_event("AUTH_REQUIRED")
        return jsonify({"error": "Authentication required"}), 401

    user = db.session.get(User, user_id)

    if user is None:
        session.clear()
        log_security_event("AUTH_REQUIRED")
        return jsonify({"error": "Authentication required"}), 401

    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({"error": "Invalid JSON"}), 400

    current_password = data.get("current_password")

    if not isinstance(current_password, str):
        return jsonify({"error": "Current password is required"}), 400

    try:
        password_hasher.verify(user.password_hash, current_password)
    except (VerifyMismatchError, VerificationError):
        log_security_event("CREDENTIAL_CHANGE_DENIED", user.id)
        return jsonify({"error": "Incorrect current password"}), 403

    new_email = data.get("new_email")
    new_password = data.get("new_password")

    if new_email is None and new_password is None:
        return jsonify({"error": "No changes requested"}), 400

    if new_email is not None:
        if not isinstance(new_email, str):
            return jsonify({"error": "Invalid email"}), 400

        new_email = new_email.strip().lower()

        if (
            len(new_email) > 255
            or new_email.count("@") != 1
            or "." not in new_email.split("@")[-1]
            or any(c.isspace() for c in new_email)
        ):
            return jsonify({"error": "Invalid email"}), 400

        existing = User.query.filter_by(email=new_email).first()

        if existing and existing.id != user.id:
            return jsonify({"error": "Email already registered"}), 409

        user.email = new_email

    if new_password is not None:
        if not isinstance(new_password, str) or not 8 <= len(new_password) <= 128:
            return jsonify({
                "error": "Password must be 8-128 characters"
            }), 400

        user.password_hash = password_hasher.hash(new_password)

    db.session.commit()
    log_security_event("CREDENTIALS_UPDATED", user.id)

    return jsonify({
        "message": "Credentials updated successfully"
    }), 200


In [ ]:
# Smoke tests: unique test user, CSRF-protected registration/login/logout, and audit events.
import uuid
email = f"test-{uuid.uuid4().hex[:10]}@example.com"
with app.test_client() as client:
    csrf_token = client.get("/api/auth/csrf-token").get_json()["csrf_token"]
    registration = client.post("/api/auth/register", json={
        "display_name": "Test User", "email": email.upper(), "password": "TestPass123!",
        "birth_year": 2000, "weight": 60, "privacy_consent": True
    }, headers={"X-CSRFToken": csrf_token})
    print("Register:", registration.status_code, registration.get_json())
    csrf_token = client.get("/api/auth/csrf-token").get_json()["csrf_token"]
    login = client.post("/api/auth/login", json={"email": email, "password": "TestPass123!"},
                        headers={"X-CSRFToken": csrf_token})
    print("Login:", login.status_code, login.get_json())
    csrf_token = client.get("/api/auth/csrf-token").get_json()["csrf_token"]
    logout = client.post("/api/auth/logout", headers={"X-CSRFToken": csrf_token})
    print("Logout:", logout.status_code, logout.get_json())
    assert (registration.status_code, login.status_code, logout.status_code) == (201, 200, 200)
with app.app_context():
    print("Recent audit events:", [row.event_type for row in SecurityLog.query.order_by(SecurityLog.id.desc()).limit(3)])


In [ ]:
# CSRF negative test: a POST without a CSRF token should be rejected.
with app.test_client() as client:
    result = client.post("/api/auth/login", json={"email": "nobody@example.com", "password": "invalid"})
    print("Without CSRF token:", result.status_code)
    assert result.status_code == 400


In [ ]:
# Test the 5-failure lockout using a fresh account.
import uuid
email = f"lock-{uuid.uuid4().hex[:10]}@example.com"
with app.test_client() as client:
    token = client.get("/api/auth/csrf-token").get_json()["csrf_token"]
    created = client.post("/api/auth/register", json={
        "display_name": "Lockout Test", "email": email, "password": "GoodPass123!",
        "birth_year": 2000, "weight": 60, "privacy_consent": True
    }, headers={"X-CSRFToken": token})
    assert created.status_code == 201
    for attempt in range(5):
        result = client.post("/api/auth/login", json={"email": email, "password": "BadPass123!"},
                             headers={"X-CSRFToken": token})
        print("Failed attempt", attempt + 1, result.status_code)
    blocked = client.post("/api/auth/login", json={"email": email, "password": "GoodPass123!"},
                          headers={"X-CSRFToken": token})
    print("Locked account:", blocked.status_code)
    assert blocked.status_code == 423
